## Deep Dive into Delta lake

In [0]:
from pyspark.sql import Row
data= [Row(id=1,name="Arun"),Row(id=2,name="vignesh")]
df=spark.createDataFrame(data)
df.show()

In [0]:
#storing as table(datawarehouse)
df.write.format("delta").mode("overwrite").saveAsTable("databricks_practice.inputdb.tblemp")
#storing as file(datalake)
df.write.format("delta").mode("overwrite").save("/databricks_practice/inputdb/empdata/empinfo")


In [0]:
%sql
select * from databricks_practice.inputdb.tblemp

In [0]:
spark.sql("DESCRIBE EXTENDED databricks_practice.inputdb.tblemp").display()

In [0]:
%sql
insert into databricks_practice.inputdb.tblemp values (3,"Saro")

In [0]:
#insert using df
df=spark.sql("select * from databricks_practice.inputdb.tblemp where id =3")
df.spark.write.format("delta").mode("append").save("dbfs:/databricks_practice/inputdb/empdata/empinfo")

##update

In [0]:
%sql
update databricks_practice.inputdb.tblemp set name ="vinoth" where id =3

In [0]:
#using df:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Load the Delta table (by catalog name)
delta_table = DeltaTable.forName(spark, "databricks_practice.inputdb.tblemp")

# Perform the update
delta_table.update(
    condition = col("id") == 34,
    set = { "name": "'vinoth'" }
)

###Delete 

In [0]:
%sql
delete from databricks_practice.inputdb.tblemp  where id =3

##Merge

In [0]:
from pyspark.sql import Row
data= [Row(id=3,name="Arun"),Row(id=2,name="Varun")]
df=spark.createDataFrame(data)
df.show()
df.createOrReplaceTempView("staging_tmp")

In [0]:
%sql
MERGE INTO databricks_practice.inputdb.tblemp AS tgt
USING staging_tmp AS src
ON tgt.id = src.id
WHEN MATCHED THEN 
  UPDATE SET tgt.name = src.name
WHEN NOT MATCHED THEN 
  INSERT (id, name) VALUES (src.id, src.name);


In [0]:
#using df api:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Load target Delta table
target = DeltaTable.forName(spark, "databricks_practice.inputdb.tblemp")

# Load or create your source DataFrame
updates_df = spark.createDataFrame([
    (34, "Vinoth"),
    (45, "Ravi")
], ["id", "name"])

# Perform MERGE
target.alias("t").merge(
    updates_df.alias("s"),
    "t.id = s.id"
).whenMatchedUpdate(set={
    "name": "s.name"
}).whenNotMatchedInsert(values={
    "id": "s.id",
    "name": "s.name"
}).execute()


In [0]:
%sql
select * from databricks_practice.inputdb.tblemp